# 1. 공정 단계별 Games–Howell 사후검정

## 요약

84개 단계·전략 쌍대비교를 한꺼번에 FDR 보정한 결과 6개가 유의했다. 유의한 차이는 모두 후기에 집중됐으며, APC는 OC보다 후기 페니실린 기울기·OUR가 높고 pH 변동·잔류 기질이 낮았다. 원본 데이터와 CSV는 변경하거나 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result


def hedges_g(sample_a, sample_b):
    n_a, n_b = len(sample_a), len(sample_b)
    pooled = ((n_a - 1) * sample_a.var(ddof=1) + (n_b - 1) * sample_b.var(ddof=1)) / (n_a + n_b - 2)
    correction = 1 - 3 / (4 * (n_a + n_b) - 9)
    return correction * (sample_a.mean() - sample_b.mean()) / np.sqrt(pooled)


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
print(f'정상 배치 {data["배치번호"].nunique()}개, 전략별 30개')

정상 배치 90개, 전략별 30개


### 판단

Fault 10개는 제외하고 RC·OC·APC 정상 배치만 비교한다. 전략별 표본 수는 같지만 분산이 다르므로 등분산을 요구하지 않는 Games–Howell 검정을 사용한다.

In [2]:
stage_ranges = {
    '초기': (0.0, 0.2), '성장기': (0.2, 0.5),
    '생산기': (0.5, 0.8), '후기': (0.8, 1.000001),
}
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    progress = time / time[-1]
    strategy = 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC')
    for stage, (lower, upper) in stage_ranges.items():
        mask = (progress >= lower) & (progress < upper)
        stage_time = time[mask]
        penicillin = batch.loc[mask, '페니실린농도(g/L)'].to_numpy()
        rows.append({
            '배치번호': batch_number, '전략': strategy, '단계': stage,
            '페니실린기울기': np.polyfit(stage_time, penicillin, 1)[0],
            'pH표준편차': batch.loc[mask, 'pH'].std(ddof=1),
            '기질평균': batch.loc[mask, '기질농도(g/L)'].mean(),
            'OUR평균': batch.loc[mask, '산소소모율(g/min)'].mean(),
            'CO2평균': batch.loc[mask, '배가스이산화탄소(%)'].mean(),
            'DO평균': batch.loc[mask, '용존산소(mg/L)'].mean(),
            '온도표준편차': batch.loc[mask, '발효온도(K)'].std(ddof=1),
        })
stage_metrics = pd.DataFrame(rows)
display(stage_metrics.groupby(['단계', '전략']).size().unstack())

전략,APC,OC,RC
단계,,,
생산기,30,30,30
성장기,30,30,30
초기,30,30,30
후기,30,30,30


### 판단

각 배치의 단계별 요약값 하나를 표본으로 사용했다. 동일 배치의 수백 개 시계열 행을 독립 표본으로 잘못 취급하는 의사 반복을 피했다. 단, 단계 진행률은 실제 종료시간을 사용하므로 이 분석은 종료 후 설명용이다.

In [3]:
metrics = ['페니실린기울기', 'pH표준편차', '기질평균', 'OUR평균', 'CO2평균', 'DO평균', '온도표준편차']
strategy_pairs = [('RC', 'OC'), ('RC', 'APC'), ('OC', 'APC')]
rows = []
for stage in stage_ranges:
    stage_data = stage_metrics.loc[stage_metrics['단계'].eq(stage)]
    for metric in metrics:
        for strategy_a, strategy_b in strategy_pairs:
            sample_a = stage_data.loc[stage_data['전략'].eq(strategy_a), metric].dropna().to_numpy()
            sample_b = stage_data.loc[stage_data['전략'].eq(strategy_b), metric].dropna().to_numpy()
            variance_a, variance_b = sample_a.var(ddof=1), sample_b.var(ddof=1)
            standard_error = np.sqrt(variance_a / len(sample_a) + variance_b / len(sample_b))
            difference = sample_a.mean() - sample_b.mean()
            df = (variance_a / len(sample_a) + variance_b / len(sample_b)) ** 2 / (
                (variance_a / len(sample_a)) ** 2 / (len(sample_a) - 1)
                + (variance_b / len(sample_b)) ** 2 / (len(sample_b) - 1)
            )
            q_statistic = abs(difference) / standard_error * np.sqrt(2)
            p_value = stats.studentized_range.sf(q_statistic, 3, df)
            q_critical = stats.studentized_range.ppf(0.95, 3, df) / np.sqrt(2)
            rows.append({
                '단계': stage, '지표': metric, '비교': f'{strategy_a}-{strategy_b}',
                '평균_A': sample_a.mean(), '평균_B': sample_b.mean(), '평균차_A-B': difference,
                '95%CI_하한': difference - q_critical * standard_error,
                '95%CI_상한': difference + q_critical * standard_error,
                'GamesHowell_p': p_value, 'Hedges_g': hedges_g(sample_a, sample_b),
            })
pairwise_results = pd.DataFrame(rows)
pairwise_results['FDR'] = fdr_bh(pairwise_results['GamesHowell_p'])
significant = pairwise_results.loc[pairwise_results['FDR'] < 0.05].sort_values('FDR')
print(f'전체 FDR 유의: {len(significant)}/{len(pairwise_results)}개')
display(significant.round(6))

전체 FDR 유의: 6/84개


,단계,지표,비교,평균_A,평균_B,평균차_A-B,95%CI_하한,95%CI_상한,GamesHowell_p,Hedges_g,FDR
65,후기,페니실린기울기,OC-APC,-0.046470,0.074248,-0.120718,-0.160089,-0.081346,0.000000,-1.908978,0.000002
68,후기,pH표준편차,OC-APC,0.032329,0.016495,0.015834,0.010008,0.021660,0.000000,1.679065,0.000005
71,후기,기질평균,OC-APC,16.641627,0.001400,16.640227,7.696062,25.584393,0.000225,1.170932,0.006293
74,후기,OUR평균,OC-APC,0.953190,1.194450,-0.241261,-0.394064,-0.088457,0.001353,-0.986945,0.028403
70,후기,기질평균,RC-APC,13.553476,0.001400,13.552077,4.397671,22.706482,0.002815,0.931725,0.047300
64,후기,페니실린기울기,RC-APC,0.008613,0.074248,-0.065635,-0.111264,-0.020006,0.003429,-0.897781,0.048006


### 최종 판단

- 84개 전체 비교에서 6개만 FDR 기준 유의했고 모두 후기에 나타났다. 생산기 옴니버스 검정의 차이는 전역 쌍대 FDR 보정 후 특정 전략 쌍까지 확정되지 않았다.
- APC 대비 OC는 후기 페니실린 기울기가 0.121 높고, OUR 평균이 0.241 높으며, pH 표준편차는 0.0158 낮고, 기질 평균은 16.64g/L 낮았다.
- APC 대비 RC도 후기 페니실린 기울기가 0.0656 높고 잔류 기질이 13.55g/L 낮았다.
- 따라서 APC 성과 차이의 가장 구체적인 후보는 후기 기질 소진, 대사활동 유지, pH 안정성과 농도 증가 유지다. 이는 연관성이지 해당 운전이 성과를 유발했다는 인과 증명은 아니다.